# Train Latent-Space Classifier

This notebook trains the logistic regression classifier used in the Hide-and-Seek pipeline.
It encodes your dataset into the DAE's semantic latent space, then trains a binary
classifier (healthy vs. malignant) on the resulting 512-dim vectors.

**Prerequisites:**
- A trained DAE checkpoint at `../checkpoints/fxclass64_autoenc/last.ckpt`
- A dataset Excel manifest with columns: `image`, `seg_msk`, `label` (0=healthy, 1=malignant)

**Output:**
- `../checkpoints/classifier_lr.pkl` — trained logistic regression classifier
- `../checkpoints/scaler.pkl` — StandardScaler fitted on all latents

## 0. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..', 'src'))

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import monai.transforms as T
import monai.data as md
from torch.utils.data import DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from templates import *

random_state = 42
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Load Pretrained DAE

In [ ]:
conf = fxclass64_autoenc()
model = LitModel(conf)

ckpt_path = os.path.join('..', 'checkpoints', conf.name, 'last.ckpt')
if not os.path.exists(ckpt_path):
    from huggingface_hub import hf_hub_download
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    hf_hub_download(
        repo_id='matanatad/dae_verts',
        filename='last.ckpt',
        local_dir=os.path.dirname(ckpt_path),
    )

state = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(state['state_dict'], strict=False)
model.ema_model.eval()
model.ema_model.to(device)
print('DAE loaded.')

## 2. Data Pipeline

Set `DATASET_PATH` to your Excel manifest.  
Required columns: `image` (NIfTI path), `seg_msk` (mask path), `label` (0/1).

In [ ]:
DATASET_PATH = '/path/to/your/dataset.xlsx'  # <-- set this

dataset_df = pd.read_excel(DATASET_PATH, engine='openpyxl')
train_df, val_df = train_test_split(dataset_df, test_size=0.15, random_state=random_state)

print(f'Total: {len(dataset_df)}, Train: {len(train_df)}, Val: {len(val_df)}')
print(f'Label distribution: {dataset_df.label.value_counts().to_dict()}')

In [ ]:
transforms = T.Compose([
    T.LoadImaged(keys=['image', 'seg_msk'], ensure_channel_first=True),
    T.CenterSpatialCropd(keys=['image', 'seg_msk'], roi_size=(-1, -1, 1)),
    T.ScaleIntensityRangePercentilesd(keys='image', lower=0, upper=99.5, b_min=0, b_max=1),
    T.SqueezeDimd(keys=['image', 'seg_msk'], dim=3, update_meta=False),
])

train_dataset = md.Dataset(data=train_df.to_dict(orient='records'), transform=transforms)
val_dataset   = md.Dataset(data=val_df.to_dict(orient='records'),   transform=transforms)
all_dataset   = md.Dataset(data=dataset_df.to_dict(orient='records'), transform=transforms)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=8)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False, num_workers=8)
all_loader   = DataLoader(all_dataset,   batch_size=1, shuffle=False, num_workers=4)

## 3. Encode Dataset to Latent Space

Pass all images through the DAE encoder to get semantic latent vectors $z \in \mathbb{R}^{512}$.

In [ ]:
def encode_dataset(loader, model, device):
    """Encode all images in a DataLoader to semantic latent vectors."""
    latents, labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Encoding'):
            img  = batch['image'].to(device)
            cond = model.encode(img, ema=True).cpu()
            latents.append(cond)
            labels.append(batch['label'][0])
    return torch.cat(latents, dim=0).numpy(), np.array(labels).ravel()

train_latents, train_labels = encode_dataset(train_loader, model, device)
val_latents,   val_labels   = encode_dataset(val_loader,   model, device)
all_latents,   all_labels   = encode_dataset(all_loader,   model, device)

print(f'Train latents: {train_latents.shape}, Val latents: {val_latents.shape}')

## 4. Train Classifier

A logistic regression classifier on the latent vectors defines the decision boundary
separating healthy from malignant vertebrae.

In [ ]:
y_train = (train_labels > 0).astype(int)
y_val   = (val_labels   > 0).astype(int)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_latents)
X_val_scaled   = scaler.transform(val_latents)

param_grid = {
    'C':            [0.001, 0.01, 0.1, 1, 5, 10],
    'penalty':      ['l1', 'l2'],
    'solver':       ['liblinear'],
    'class_weight': [None, 'balanced'],
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=random_state),
    param_grid, cv=5, scoring='f1', n_jobs=-1,
)
grid.fit(X_train_scaled, y_train)

best_lr  = grid.best_estimator_
y_pred   = best_lr.predict(X_val_scaled)
y_scores = best_lr.decision_function(X_val_scaled)

print(f'Best params:  {grid.best_params_}')
print(f'Val Accuracy: {accuracy_score(y_val, y_pred):.3f}')
print(f'Val F1:       {f1_score(y_val, y_pred):.3f}')
print(f'Val AUC:      {roc_auc_score(y_val, y_scores):.3f}')

## 5. Retrain on All Data and Save

After selecting hyperparameters via cross-validation, retrain on the full dataset
to maximize coverage before saving.

In [ ]:
y_all = (all_labels > 0).astype(int)
scaler_all   = StandardScaler()
X_all_scaled = scaler_all.fit_transform(all_latents)

lr_model = LogisticRegression(
    max_iter=1000, random_state=random_state,
    **{k: v for k, v in grid.best_params_.items()},
)
lr_model.fit(X_all_scaled, y_all)

train_acc = accuracy_score(y_all, lr_model.predict(X_all_scaled))
print(f'Train accuracy (all data): {train_acc:.3f}')

os.makedirs(os.path.join('..', 'checkpoints'), exist_ok=True)
joblib.dump(lr_model,   os.path.join('..', 'checkpoints', 'classifier_lr.pkl'))
joblib.dump(scaler_all, os.path.join('..', 'checkpoints', 'scaler.pkl'))
print('Saved classifier and scaler to checkpoints/')